# Phase 1 — reconstruction-stability segmentation

**Runtime → Change runtime type → T4 GPU** before running anything.

All state is checkpointed to Google Drive every 5 series. A quota cutoff or VM recycle
costs nothing: reopen, run step 0, run the segmentation cell, and it resumes.


## 0. Drive + state (run first, every session)


In [ ]:
# STEP 0 — Drive first. All durable state lives in Drive, never on the VM.
# Colab recycles VMs without warning; anything only in /content is temporary.
from google.colab import drive
import os, shutil, glob
from pathlib import Path
drive.mount("/content/drive")
D = Path("/content/drive/MyDrive/phase1_imaging"); D.mkdir(parents=True, exist_ok=True)
os.chdir("/content")

# restore any prior state
for f in ["volumes.csv","colab_series_manifest.csv","colab_pairs.csv","model_provenance.json"]:
    if (D/f).exists() and not Path(f).exists():
        shutil.copy(D/f, f); print("restored from Drive:", f)

# manifests are small and required — upload once, then they persist in Drive
if not Path("colab_series_manifest.csv").exists():
    print("\nUpload colab_series_manifest.csv and colab_pairs.csv:")
    from google.colab import files
    files.upload()
    for f in ["colab_series_manifest.csv","colab_pairs.csv"]:
        if Path(f).exists(): shutil.copy(f, D)

import pandas as pd
man = pd.read_csv("colab_series_manifest.csv")
if Path("volumes.csv").exists():
    v = pd.read_csv("volumes.csv")
    ok = v[v.error.isna() | v.error.astype(str).str.strip().eq("")]
    print(f"\nstate: {len(v)}/{len(man)} attempted | {len(ok)} ok | {len(v)-len(ok)} failed | {len(man)-len(v)} remaining")
else:
    print(f"\nstate: nothing done yet | {len(man)} series to process")

## 1. Environment — pin everything

In [ ]:
!pip -q install TotalSegmentator==2.18.0 pydicom==3.0.1 nibabel==5.3.2 SimpleITK==2.4.1 2>&1 | tail -3
import subprocess, torch, json, os
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "Enable the GPU runtime before continuing."
ver = subprocess.run(["TotalSegmentator","--version"],capture_output=True,text=True).stdout.strip()
print(ver)

## 3. Record the exact model provenance

The pre-specification requires the model version and weights hash to be logged. An
unpinned run is discarded, so this runs *before* segmentation and its output is saved
alongside the results.

In [ ]:
# Provenance. Writes model_provenance.json and copies it to Drive.
# Weight hashes are filled in by the 4b cell AFTER the first segmentation,
# because TotalSegmentator downloads weights on first use, not on install.
import json, os, torch, shutil
from pathlib import Path
from importlib.metadata import version as pkgver
D = Path("/content/drive/MyDrive/phase1_imaging")

prov = json.load(open("model_provenance.json")) if Path("model_provenance.json").exists() else {}
prov.update({"totalsegmentator": pkgver("TotalSegmentator"), "nnunetv2": pkgver("nnunetv2"),
             "torch": torch.__version__,
             "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
             "task": "total",
             "structures": ["heart","aorta","pulmonary_vein","superior_vena_cava","inferior_vena_cava"],
             "stats_flags": "-s -sx -sii -ss (statistics incl. border-touching structures; masks not saved)"})
wdir = Path(os.environ["TOTALSEG_WEIGHTS_PATH"]) if "TOTALSEG_WEIGHTS_PATH" in os.environ \
       else Path(os.environ.get("TOTALSEG_HOME_DIR", Path.home()/".totalsegmentator"))/"nnunet/results"
prov["weights_dir"] = str(wdir)
json.dump(prov, open("model_provenance.json","w"), indent=2)
if D.exists(): shutil.copy("model_provenance.json", D)
print(json.dumps(prov, indent=1))

## 4. Download → segment → discard

One series at a time. The DICOM volume is deleted immediately after its masks are written,
so peak disk stays low regardless of the 20 GB total. Progress is appended to
`volumes.csv` after every series, so a disconnect costs at most one series.

## 3b. Smoke test — one series

Catches flag and output-path errors in ~2 minutes instead of 217 failures.


In [ ]:
# Run ONE series first and inspect it before committing to 217.
import subprocess, json, shutil, urllib.request, urllib.parse, zipfile, io, pandas as pd
from pathlib import Path
man = pd.read_csv("colab_series_manifest.csv")
STRUCTS = ["heart","aorta","pulmonary_vein","superior_vena_cava","inferior_vena_cava"]
uid = man.sort_values("n_images").iloc[0].series_uid
dcm, outdir = Path("smoke_dcm"), Path("smoke_out")
shutil.rmtree(dcm, ignore_errors=True); shutil.rmtree(outdir, ignore_errors=True)
dcm.mkdir(parents=True, exist_ok=True); outdir.mkdir(parents=True, exist_ok=True)
url = "https://services.cancerimagingarchive.net/nbia-api/services/v1/getImage?" + \
      urllib.parse.urlencode({"SeriesInstanceUID": uid})
with urllib.request.urlopen(url, timeout=900) as r:
    zipfile.ZipFile(io.BytesIO(r.read())).extractall(dcm)
cp = subprocess.run(["TotalSegmentator","-i",str(dcm),"-o",str(outdir),
                     "--task","total","--roi_subset",*STRUCTS,
                     "-s","-sx","-sii","-ss","--quiet"], capture_output=True, text=True, timeout=2400)
print("exit", cp.returncode, "| files:", [p.name for p in outdir.iterdir()])
if cp.returncode: print((cp.stderr or cp.stdout)[-800:])
st = json.load(open(outdir/"statistics.json"))
for s in STRUCTS:
    d = st.get(s, {})
    print(f"  {s:22s} vol={d.get('volume')} mm3  nvox={d.get('n_voxels')}")
assert (st.get("heart",{}).get("volume") or 0) > 0, "heart volume is 0 — investigate before the full run"
shutil.rmtree(dcm, ignore_errors=True); shutil.rmtree(outdir, ignore_errors=True)
print("\nSMOKE TEST PASSED — proceed to the full run.")

## 3d. Retry failed series (optional)

Run once before resuming if you want the earlier failures reattempted.


In [ ]:
# Drop failed rows so the next run retries them with the gantry-tilt fix and longer timeout.
import pandas as pd, shutil
from pathlib import Path
D = Path("/content/drive/MyDrive/phase1_imaging")
v = pd.read_csv("volumes.csv")
bad = v.error.notna() & v.error.astype(str).str.strip().ne("")
print(f"dropping {int(bad.sum())} failed rows; keeping {int((~bad).sum())} successful")
print(v.loc[bad,"error"].str.slice(0,60).value_counts().to_string() if bad.any() else "(none)")
v[~bad].to_csv("volumes.csv", index=False); shutil.copy("volumes.csv", D)

In [ ]:
# Segmentation with full error logs, gantry-tilt handling, and Drive checkpointing.
import urllib.request, urllib.parse, json, zipfile, io, shutil, time, subprocess, os
import pandas as pd
from pathlib import Path

D = Path("/content/drive/MyDrive/phase1_imaging")   # mounted in the restore cell
man = pd.read_csv("colab_series_manifest.csv")
BASE = "https://services.cancerimagingarchive.net/nbia-api/services/v1"
STRUCTS = ["heart","aorta","pulmonary_vein","superior_vena_cava","inferior_vena_cava"]
CSV = Path("volumes.csv"); LOGS = Path("logs"); LOGS.mkdir(exist_ok=True)

done = set(pd.read_csv(CSV).series_uid) if CSV.exists() else set()
print(f"resuming — {len(done)} attempted already")

# Gantry-tilted CT fails dicom2nifti's orthogonality check. Resample instead of
# skipping: both members of a pair share the acquisition, so tilt is identical and
# resampling is applied symmetrically. Series needing it are flagged.
PRE = ("import dicom2nifti.settings as s; "
       "s.disable_validate_orthogonal(); s.disable_validate_slice_increment(); "
       "s.enable_resampling(); s.set_resample_spline_interpolation_order(1); "
       "s.set_resample_padding(-1000)")

def fetch(uid, dest):
    url = f"{BASE}/getImage?" + urllib.parse.urlencode({"SeriesInstanceUID": uid})
    with urllib.request.urlopen(url, timeout=900) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    dest.mkdir(parents=True, exist_ok=True); z.extractall(dest)
    return sum(1 for n in z.namelist() if n.lower().endswith(".dcm"))

def run_ts(dcm, outdir, lenient):
    cmd = ["TotalSegmentator","-i",str(dcm),"-o",str(outdir),"--task","total",
           "--roi_subset",*STRUCTS,"-s","-sx","-sii","-ss","--quiet"]
    if lenient:
        code = f"{PRE}\nimport sys; sys.argv={cmd!r}\n" \
               "from totalsegmentator.bin.TotalSegmentator import main; main()"
        return subprocess.run(["python","-c",code], capture_output=True, text=True, timeout=3600)
    return subprocess.run(cmd, capture_output=True, text=True, timeout=3600)

for i, row in man.iterrows():
    uid = row.series_uid
    if uid in done: continue
    t0 = time.time(); dcm, outdir = Path("tmp_dcm"), Path("tmp_out")
    shutil.rmtree(dcm, ignore_errors=True); shutil.rmtree(outdir, ignore_errors=True)
    outdir.mkdir(parents=True, exist_ok=True)
    rec = {"series_uid": uid, "patient": row.patient, "error": "", "resampled": False}
    try:
        n = fetch(uid, dcm)
        cp = run_ts(dcm, outdir, lenient=False)
        if cp.returncode != 0 and "NON_CUBICAL" in (cp.stderr or "")+(cp.stdout or ""):
            cp = run_ts(dcm, outdir, lenient=True); rec["resampled"] = True
        # keep the FULL log — a 300-char tail catches the semaphore warning, not the error
        (LOGS/f"{uid[-16:]}.log").write_text(f"RC={cp.returncode}\n--STDOUT--\n{cp.stdout}\n--STDERR--\n{cp.stderr}")
        sf = outdir/"statistics.json"
        if not sf.exists():
            err = [l for l in (cp.stderr or "").splitlines()
                   if ("Error" in l or "error" in l) and "Warning" not in l]
            raise RuntimeError(f"rc={cp.returncode} " + (" | ".join(err[-2:]) if err else "no statistics.json"))
        stats = json.load(open(sf))
        for s in STRUCTS:
            d = stats.get(s, {})
            rec[f"{s}_vol_mm3"] = d.get("volume"); rec[f"{s}_nvox"] = d.get("n_voxels")
            rec[f"{s}_intensity"] = d.get("intensity")
        rec["n_slices"] = n
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {str(e)[:220]}"
        print(f"  !! {uid[-12:]}: {rec['error']}", flush=True)
    rec["runtime_s"] = round(time.time()-t0,1)
    pd.DataFrame([rec]).to_csv(CSV, mode="a", header=not CSV.exists(), index=False)
    shutil.rmtree(dcm, ignore_errors=True); shutil.rmtree(outdir, ignore_errors=True)
    if (i+1) % 5 == 0 and D.exists():
        shutil.copy(CSV, D)          # checkpoint to Drive every 5 series
    if (i+1) % 10 == 0:
        v = pd.read_csv(CSV)
        nf = int((v.error.notna() & v.error.astype(str).str.strip().ne("")).sum())
        print(f"{i+1}/{len(man)} | done {len(v)} | failed {nf} | last {rec['runtime_s']}s", flush=True)

if D.exists(): shutil.copy(CSV, D); shutil.make_archive(str(D/"logs"),"zip","logs")
v = pd.read_csv(CSV)
print(f"\ncomplete: {len(v)}/{len(man)}")

## 4b. Hash the weights

Deferred until here because TotalSegmentator downloads its weights on first use, not on install.


In [ ]:
# Run AFTER at least one series has segmented, so the weights exist on disk.
import hashlib, json, shutil
from pathlib import Path
D = Path("/content/drive/MyDrive/phase1_imaging")
def sha256(p, buf=1<<20):
    h = hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda: f.read(buf), b""): h.update(b)
    return h.hexdigest()
prov = json.load(open("model_provenance.json"))
wdir = Path(prov["weights_dir"])
ckpt = sorted(p for ext in ("*.pth","*.pt","*.model") for p in wdir.rglob(ext) if p.is_file())
assert ckpt, f"no weight files under {wdir} — has the segmentation cell run?"
prov["weights"] = {str(p.relative_to(wdir)): sha256(p)[:32] for p in ckpt}
prov["n_weight_files"] = len(ckpt)
prov["weights_total_MiB"] = round(sum(p.stat().st_size for p in ckpt)/2**20, 1)
json.dump(prov, open("model_provenance.json","w"), indent=2)
if D.exists(): shutil.copy("model_provenance.json", D)
print(f"hashed {len(ckpt)} checkpoint(s), {prov['weights_total_MiB']} MiB")

## 5. Endpoint 3 — segmentation failure rate

Pre-specified as reportable regardless of what endpoints 1 and 2 show.

In [ ]:
# Endpoint 3 — failure rate, pre-specified as reportable regardless of endpoints 1 and 2.
import pandas as pd, shutil
from pathlib import Path
D = Path("/content/drive/MyDrive/phase1_imaging")
man = pd.read_csv("colab_series_manifest.csv"); v = pd.read_csv("volumes.csv")
v["failed"] = v.error.notna() & v.error.astype(str).str.strip().ne("")
m = v.merge(man[["series_uid","vendor","slice_thickness_mm"]], on="series_uid", how="left")
print("failure rate: %d/%d (%.1f%%)" % (m.failed.sum(), len(m), 100*m.failed.mean()))
print("\nby vendor:");    print(m.groupby("vendor").failed.agg(["sum","count","mean"]).to_string())
print("\nby thickness:"); print(m.groupby("slice_thickness_mm").failed.agg(["sum","count","mean"]).to_string())
if "resampled" in m: print("\nresampled (gantry tilt):", int(m.resampled.fillna(False).sum()))
if m.failed.any():
    print("\nerrors:"); print(m.loc[m.failed,"error"].str.slice(0,70).value_counts().to_string())
m.to_csv("failure_rates.csv", index=False)
if D.exists(): shutil.copy("failure_rates.csv", D)

## 6. Download the results

Three small files go back to the analysis session — no images, no volumes. Endpoints 1
and 2 are computed there, against the pre-specified aggregation rules.

In [ ]:
import os
from google.colab import files
for f in ["volumes.csv","failure_rates.csv","model_provenance.json"]:
    if os.path.exists(f): files.download(f)
    else: print("MISSING:", f)